In [4]:
import json
import re
import requests
import os
from typing import List

from prompts import MAIN_SYSTEM_PROMPT, MAIN_USER_PROMPT


# ==========================================
# 1️⃣  DeepSeek LLM 调用
# ==========================================
def call_llm(
    model: str,
    api_key: str,
    user_prompt: str,
    system_prompt: str,
    base_url: str = "https://api.deepseek.com/v1",
    temperature: float = 0,
    max_tokens: int = 2000,
):
    url = f"{base_url}/chat/completions"

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
    }

    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        "temperature": temperature,
        "max_tokens": max_tokens,
    }

    response = requests.post(url, headers=headers, json=payload)
    response.raise_for_status()

    return response.json()["choices"][0]["message"]["content"]


# ==========================================
# 2️⃣  文本分块
# ==========================================
def chunk_text(text: str, chunk_size: int = 500) -> List[str]:
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end
    return chunks


# ==========================================
# 3️⃣  提取 JSON
# ==========================================
def extract_json(text: str):
    json_pattern = r"\[.*?\]"
    match = re.search(json_pattern, text, re.DOTALL)
    if not match:
        return None

    try:
        return json.loads(match.group())
    except Exception:
        return None


# ==========================================
# 4️⃣  实体标准化
# ==========================================
STANDARDIZE_SYSTEM_PROMPT = """
你是一个知识图谱实体标准化助手。
1. 合并同义实体
2. 统一表达形式
3. 不改变语义
4. 输出保持 JSON 数组格式
"""

def standardize_entities(triples, api_key):
    if not triples:
        return []

    user_prompt = f"""
以下是因果三元组，请对实体进行标准化：

{json.dumps(triples, ensure_ascii=False, indent=2)}

输出格式保持不变。
"""

    raw_output = call_llm(
        model="deepseek-chat",
        api_key=api_key,
        user_prompt=user_prompt,
        system_prompt=STANDARDIZE_SYSTEM_PROMPT,
    )

    normalized = extract_json(raw_output)
    return normalized if normalized else triples


# ==========================================
# 5️⃣  抽取单个 chunk
# ==========================================
def extract_from_chunk(text: str, api_key: str):
    user_prompt = MAIN_USER_PROMPT + f"```\n{text}\n```"

    raw_output = call_llm(
        model="deepseek-chat",
        api_key=api_key,
        user_prompt=user_prompt,
        system_prompt=MAIN_SYSTEM_PROMPT,
    )

    triples = extract_json(raw_output)
    if not triples:
        return []

    clean_triples = []
    for item in triples:
        if (
            isinstance(item, dict)
            and "cause" in item
            and "relationship_type" in item
            and "effect" in item
        ):
            clean_triples.append(item)

    return clean_triples


# ==========================================
# 6️⃣  单文件处理
# ==========================================
def process_file(filepath: str, api_key: str):
    print(f"\n📄 处理文件: {filepath}")

    with open(filepath, "r", encoding="utf-8") as f:
        text = f.read()

    chunks = chunk_text(text)
    all_triples = []

    for i, chunk in enumerate(chunks):
        print(f"   🔹 处理 chunk {i+1}/{len(chunks)}")
        triples = extract_from_chunk(chunk, api_key)
        all_triples.extend(triples)

    print("   🔧 实体标准化中...")
    all_triples = standardize_entities(all_triples, api_key)

    return all_triples


# ==========================================
# 7️⃣  主程序：遍历 texts 目录
# ==========================================
if __name__ == "__main__":
    api_key = os.getenv("DEEPSEEK_API_KEY")
    if not api_key:
        raise ValueError("请设置环境变量 DEEPSEEK_API_KEY")

    texts_dir = "texts"

    if not os.path.exists(texts_dir):
        raise ValueError("未找到 texts 文件夹")

    all_results = {}

    for filename in os.listdir(texts_dir):
        if filename.endswith(".txt"):
            filepath = os.path.join(texts_dir, filename)
            triples = process_file(filepath, api_key)
            all_results[filename] = triples

            print("\n===== 结果 =====")
            print(json.dumps(triples, indent=2, ensure_ascii=False))

    # 可选：保存汇总结果
    with open("all_triples.json", "w", encoding="utf-8") as f:
        json.dump(all_results, f, indent=2, ensure_ascii=False)

    print("\n✅ 所有文件处理完成，结果已保存到 all_triples.json")


📄 处理文件: texts\1.txt
   🔹 处理 chunk 1/3
   🔹 处理 chunk 2/3
   🔹 处理 chunk 3/3
   🔧 实体标准化中...

===== 结果 =====
[
  {
    "cause": "掷杯不中",
    "effect": "仪式中断",
    "relationship_type": "导致（风险）"
  },
  {
    "cause": "孩童登轿",
    "effect": "视为新童",
    "relationship_type": "导致"
  },
  {
    "cause": "许某蓝拒绝巡游",
    "effect": "引发短暂争论",
    "relationship_type": "导致"
  },
  {
    "cause": "村民邀请劝说",
    "effect": "许某蓝登轿",
    "relationship_type": "导致"
  },
  {
    "cause": "许某蓝登轿",
    "effect": "完成巡游仪式",
    "relationship_type": "导致"
  },
  {
    "cause": "网传许某涵为男孩",
    "effect": "通报澄清性别",
    "relationship_type": "导致（风险）"
  },
  {
    "cause": "等待上轿",
    "effect": "暂停推轿",
    "relationship_type": "导致"
  }
]

✅ 所有文件处理完成，结果已保存到 all_triples.json
